#1. Tạo dữ liệu & kết nối bảng

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng student
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

# Tạo bảng course
cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

In [2]:
students = [
(1,'Nguyen Minh Hoang','May Tinh',12,6.7),
(2,'Tran Thi Lan','Kinh Te',34,9.2),
(3,'Pham Van Nam','Toan Tin',None,7.9),
(4,'Le Thanh Huyen','Toan Tin',20,7.2),
(5,'Vu Quoc Anh','May Tinh',24,8.0),
(6,'Dang Thuy Linh','May Tinh',24,5.5),
(7,'Bui Tien Dung','Kinh Te',34,9.2),
(8,'Ho Ngoc Mai','Toan Tin',20,8.8),
(9,'Duong Huu Phuc','Kinh Te',None,7.2),
(10,'Cao Thi Hanh','May Tinh',None,7.0)
]

courses = [
(12,'Giai tich'),
(34,'Thong ke'),
(26,'Tin hoc')
]

cursor.executemany("INSERT INTO student VALUES (?,?,?,?,?)", students)
cursor.executemany("INSERT INTO course VALUES (?,?)", courses)

conn.commit()

In [3]:
pd.read_sql("""
SELECT * FROM student, course
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


In [4]:
pd.read_sql("""
SELECT *
FROM student s
INNER JOIN course c
ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


In [5]:
pd.read_sql("""
SELECT *
FROM student s
LEFT JOIN course c
ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


In [6]:
pd.read_sql("""
SELECT *
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
""", conn)

,id,course_name,student_id,name,class,course_id,score
0,12,Giai tich,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,34,Thong ke,2.0,Tran Thi Lan,Kinh Te,34.0,9.2
2,34,Thong ke,7.0,Bui Tien Dung,Kinh Te,34.0,9.2
3,26,Tin hoc,NaN,None,None,NaN,NaN


In [7]:
pd.read_sql("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,None,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,None,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,None,7.0,NaN,None


#2. Xử lý dữ liệu

In [8]:
cursor.execute("""
UPDATE student
SET course_id = 26
WHERE course_id IS NULL
""")
conn.commit()

In [9]:
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

In [10]:
#2.a
pd.read_sql("""
SELECT class,
       COUNT(*) AS total_students,
       ROUND(AVG(score),2) AS avg_score
FROM student
GROUP BY class
""", conn)

,class,total_students,avg_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


In [11]:
#2.b
pd.read_sql("""
SELECT c.course_name,
       COUNT(*) AS total_students,
       ROUND(AVG(score),2) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

,course_name,total_students,avg_score
0,Giai tich,1,6.70
1,Thong ke,2,9.20
2,Tin hoc,3,7.37


In [13]:
#2.c
pd.read_sql("""
SELECT c.course_name,
       ROUND(AVG(score),2) AS avg_score,
       CASE
           WHEN AVG(score) >= 9 THEN 'Xuat sac'
           WHEN AVG(score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS ranking
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

,course_name,avg_score,ranking
0,Giai tich,6.70,Tot
1,Thong ke,9.20,Xuat sac
2,Tin hoc,7.37,Tot


#3. Xếp hạng sinh viên

In [14]:
#3.a
pd.read_sql("""
SELECT *,
       RANK() OVER (ORDER BY score DESC) AS rank
FROM student
""", conn)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


In [15]:
#3.b
pd.read_sql("""
SELECT *,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
FROM student
""", conn)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


In [16]:
#3.c
pd.read_sql("""
SELECT *,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank
FROM student
""", conn)

,student_id,name,class,course_id,score,rank
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,26,7.9,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,10,Cao Thi Hanh,May Tinh,26,7.0,3
4,2,Tran Thi Lan,Kinh Te,34,9.2,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,1


In [17]:
# top 3 cao nhất
pd.read_sql("""
SELECT * FROM (
    SELECT *, RANK() OVER (ORDER BY score DESC) r
    FROM student
)
WHERE r <= 3
""", conn)

,student_id,name,class,course_id,score,r
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3


In [18]:
# top 3 thấp nhất
pd.read_sql("""
SELECT * FROM (
    SELECT *, RANK() OVER (ORDER BY score ASC) r
    FROM student
)
WHERE r <= 3
""", conn)

,student_id,name,class,course_id,score,r
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3


#4. Thêm graduation_date

In [19]:
cursor.execute("""
ALTER TABLE student
ADD COLUMN graduation_date DATETIME
""")

In [20]:
cursor.execute("""
UPDATE student
SET graduation_date = datetime('now', '+' || CAST(score AS INTEGER) || ' days')
""")
conn.commit()

In [21]:
df = pd.read_sql("""
SELECT student_id, name, score, graduation_date
FROM student
""", conn)

df.head()

,student_id,name,score,graduation_date
0,1,Nguyen Minh Hoang,6.7,2026-04-09 00:30:27
1,2,Tran Thi Lan,9.2,2026-04-12 00:30:27
2,3,Pham Van Nam,7.9,2026-04-10 00:30:27
3,7,Bui Tien Dung,9.2,2026-04-12 00:30:27
4,9,Duong Huu Phuc,7.2,2026-04-10 00:30:27


# KẾT LUẬN

## 1. Về kết nối dữ liệu giữa hai bảng
- Đã thực hiện các phương pháp:
  - CROSS JOIN: tạo tất cả tổ hợp giữa hai bảng
  - INNER JOIN: chỉ giữ các bản ghi hợp lệ
  - LEFT JOIN: giữ toàn bộ sinh viên
  - RIGHT JOIN, FULL OUTER JOIN: được mô phỏng trong SQLite
- Kết luận:
  - INNER JOIN dùng khi cần dữ liệu chính xác
  - LEFT JOIN phù hợp khi cần giữ toàn bộ dữ liệu

## 2. Về xử lý và làm sạch dữ liệu
- Đã thực hiện:
  - Cập nhật các giá trị course_id bị thiếu
  - Loại bỏ các bản ghi không hợp lệ
- Sau xử lý:
  - Dữ liệu đồng nhất và hợp lệ
  - Không còn giá trị NULL hoặc sai khóa ngoại

## 3. Về thống kê dữ liệu
### a. Theo lớp học
- Đã tính được:
  - Tổng số sinh viên từng lớp
  - Điểm trung bình từng lớp
- Nhận xét:
  - Có sự chênh lệch điểm giữa các lớp
### b. Theo môn học
- Đã tính được:
  - Tổng số sinh viên theo từng môn
  - Điểm trung bình từng môn
- Nhận xét:
  - Một số môn có nhiều sinh viên và điểm cao hơn
### c. Phân loại thi đua
- Phân loại theo tiêu chí:
  - ≥ 9.0 → Xuất sắc
  - 6.0 – 8.9 → Tốt
  - < 6.0 → Kém
- Kết luận:
  - Phần lớn sinh viên đạt mức Tốt
  - Một số ít đạt Xuất sắc hoặc Kém

## 4. Về xếp hạng sinh viên
- Đã sử dụng hàm RANK() để xếp hạng:
  - Toàn bộ sinh viên
  - Theo lớp học
  - Theo môn học
- Đã xác định:
  - Top 3 sinh viên điểm cao nhất
  - Top 3 sinh viên điểm thấp nhất
- Kết luận:
  - Có sự phân hóa rõ ràng giữa các nhóm sinh viên

## 5. Về bổ sung trường graduation_date
- Đã thêm trường graduation_date (DATETIME)
- Thời gian tốt nghiệp được tính:
  - Thời điểm hiện tại + số ngày tương ứng với điểm số
- Kết quả:
  - Tất cả sinh viên đều có dữ liệu tốt nghiệp
  - Dữ liệu có thể dùng cho phân tích thời gian học tập

## 6. Đánh giá chung
- Đã hoàn thành đầy đủ các yêu cầu:
  - Kết nối dữ liệu
  - Làm sạch dữ liệu
  - Thống kê và phân tích
  - Xếp hạng
  - Mở rộng dữ liệu
- Kết quả đạt được:
  - Dữ liệu chính xác, đầy đủ
  - Có giá trị ứng dụng thực tế trong quản lý và phân tích